# Mejora 1 - Modelo en dos etapas por oración

**Proyecto - Parte 1: Machine Learning clásico**

**Estudiantes:**

[Nombre integrante 1] – [código]

[Nombre integrante 2] – [código]

**Curso:** ISIS2611 - Aprendizaje de Máquina

---

## Contexto

En el notebook principal (`Proyecto_Parte1_ML_Clasico.ipynb`) el mejor modelo fue una regresión logística sobre tres bloques de TF-IDF (texto completo, última oración y cláusula final), con una exactitud de 0,895 y un F1-macro de 0,899 en el conjunto de prueba. El análisis de errores de ese notebook mostró que casi todos los errores restantes están en reseñas positivas y negativas, y que la etiqueta suele depender de la última opinión de la reseña.

Este notebook prueba la primera de las oportunidades de mejora que quedaron identificadas: **localizar mejor la opinión que define la etiqueta**. Para eso se parte cada reseña en trozos, se puntúa cada trozo con un primer modelo y un segundo modelo decide la etiqueta a partir de esos puntajes.

## Requisitos de la guía

| Requisito | Cómo se cumple |
|:---|:---|
| No usar aprendizaje profundo, transformers ni embeddings preentrenados. | No se usa ninguno. Todo se basa en TF-IDF y n-gramas. |
| Usar únicamente modelos de scikit-learn. | Los modelos que componen el sistema son `LogisticRegression` y `MultinomialNB`. La clase `ModeloDosEtapas` solo los encadena con la interfaz de scikit-learn (`fit` y `predict`). |
| Resultados replicables. | Semillas fijas, versiones en `requirements.txt` y todo el flujo en este notebook (sección 12). |

Encadenar dos modelos es una idea que va un paso más allá de lo visto en clase. Si el equipo considera que no encaja con "técnicas vistas en el curso", el modelo del notebook principal sigue siendo la opción más simple.

## 1. Importación de librerías

In [1]:
import os
import re
import unicodedata
import warnings
from collections import Counter

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from nltk.stem import SnowballStemmer

from sklearn.base import BaseEstimator, ClassifierMixin, clone
from sklearn.compose import ColumnTransformer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, classification_report, ConfusionMatrixDisplay,
                             f1_score, precision_score, recall_score)
from sklearn.model_selection import GridSearchCV, StratifiedKFold, cross_val_score, train_test_split
from sklearn.naive_bayes import MultinomialNB
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer, StandardScaler

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid')
pd.set_option('display.max_colwidth', 120)
pd.set_option('display.width', 160)

RANDOM_STATE = 42
ORDEN_CLASES = ['negativo', 'neutral', 'positivo']

# Rutas del proyecto (este notebook está en la carpeta notebooks/)
RUTA_DATOS = os.path.join('..', 'data')
RUTA_MODELOS = os.path.join('..', 'models')
RUTA_ENVIOS = os.path.join('..', 'submissions')
NOMBRE_MODELO = 'dos_etapas_por_oracion'

os.makedirs(RUTA_MODELOS, exist_ok=True)
os.makedirs(RUTA_ENVIOS, exist_ok=True)

## 2. Datos y partición

Se usan los mismos datos, la misma partición 80/20 estratificada (semilla 42) y las mismas 5 particiones de validación cruzada del notebook principal, de modo que los resultados son comparables.

In [2]:
train = pd.read_csv(os.path.join(RUTA_DATOS, 'train.csv'))
evaluacion = pd.read_csv(os.path.join(RUTA_DATOS, 'eval.csv'))
muestra_envio = pd.read_csv(os.path.join(RUTA_DATOS, 'sample_submission.csv'))

print('train:', train.shape, '| eval:', evaluacion.shape, '| sample_submission:', muestra_envio.shape)
print('Distribución de clases en train:')
display(train['label'].value_counts(normalize=True).reindex(ORDEN_CLASES).round(3).to_frame('proporción'))

train: (12000, 3) | eval: (3000, 2) | sample_submission: (3000, 2)
Distribución de clases en train:


,proporción
label,
negativo,0.351
neutral,0.298
positivo,0.351


In [3]:
X = train[['text']]
y = train['label']

# Misma partición y mismas particiones de validación cruzada que en el notebook principal
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)
kfold = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

print('Entrenamiento:', X_train.shape, '| Prueba:', X_test.shape)

Entrenamiento: (9600, 1) | Prueba: (2400, 1)


## 3. Preprocesamiento y modelo actual de referencia

Se repiten las funciones de preprocesamiento y la representación por bloques del notebook principal. Se usan para el modelo actual, que sirve de referencia, y como parte de la etapa 2. La referencia obtiene un F1-macro de 0,8992 en validación cruzada y una exactitud de 0,8946 y un F1-macro de 0,8993 en el conjunto de prueba.

In [4]:
stemmer = SnowballStemmer('spanish')
cache_raices = {}


def raiz(palabra):
    # Se guarda la raíz de cada palabra para no repetir el stemming miles de veces
    if palabra not in cache_raices:
        cache_raices[palabra] = stemmer.stem(palabra)
    return cache_raices[palabra]


def quitar_tildes(texto):
    texto = unicodedata.normalize('NFKD', texto)
    return ''.join(c for c in texto if not unicodedata.combining(c))


def limpiar_texto(texto, stop=None, stemming=True):
    texto = quitar_tildes(texto.lower())
    texto = re.sub(r'[^\w\s]', ' ', texto)
    texto = re.sub(r'\d+', ' ', texto)
    tokens = texto.split()
    if stop:
        tokens = [t for t in tokens if t not in stop]
    if stemming:
        tokens = [raiz(t) for t in tokens]
    return ' '.join(tokens)


def separar_oraciones(texto):
    return [o.strip() for o in re.split(r'(?<=[.!?])\s+', texto) if o.strip()]


# Conectores que introducen la opinión que se impone sobre la anterior
CONECTORES = r'\b(?:pero|aunque|sin embargo|aun asi|con todo|a fin de cuentas|al final)\b'


def extraer_completo(textos, stemming=True):
    return [limpiar_texto(t, stemming=stemming) for t in textos]


def extraer_ultima(textos, stemming=True):
    return [limpiar_texto(separar_oraciones(t)[-1], stemming=stemming) for t in textos]


def extraer_clausula_final(textos, stemming=True):
    salida = []
    for t in textos:
        ultima = quitar_tildes(separar_oraciones(t)[-1].lower())
        salida.append(limpiar_texto(re.split(CONECTORES, ultima)[-1], stemming=stemming))
    return salida


def bloque_tfidf(nombre, extractor, rango=(1, 2)):
    return (nombre,
            Pipeline([('extraer', FunctionTransformer(extractor)),
                      ('tfidf', TfidfVectorizer(ngram_range=rango))]),
            'text')


def construir_representacion(bloques=('completo', 'ultima', 'clausula'), rango=(1, 2)):
    extractores = {'completo': extraer_completo,
                   'ultima': extraer_ultima,
                   'clausula': extraer_clausula_final}
    return ColumnTransformer([bloque_tfidf(b, extractores[b], rango) for b in bloques])

In [5]:
def metricas(y_real, y_pred):
    return {'Exactitud': accuracy_score(y_real, y_pred),
            'Precisión (macro)': precision_score(y_real, y_pred, average='macro'),
            'Recall (macro)': recall_score(y_real, y_pred, average='macro'),
            'F1 (macro)': f1_score(y_real, y_pred, average='macro')}

In [6]:
# Modelo actual (el del notebook principal): regresión logística L1 sobre la representación por bloques
modelo_actual = Pipeline([
    ('rep', construir_representacion()),
    ('modelo', LogisticRegression(C=10, penalty='l1', solver='liblinear', max_iter=3000, random_state=RANDOM_STATE)),
])
modelo_actual.fit(X_train, y_train)
pred_actual = pd.Series(modelo_actual.predict(X_test), index=y_test.index)

cv_actual = cross_val_score(modelo_actual, X_train, y_train, cv=kfold, scoring='f1_macro', n_jobs=-1)
print(f'Modelo actual - F1-macro en validación cruzada (train): {cv_actual.mean():.4f} +/- {cv_actual.std():.4f}')
pd.DataFrame({'Modelo actual (test)': metricas(y_test, pred_actual)}).T.round(4)

Modelo actual - F1-macro en validación cruzada (train): 0.8992 +/- 0.0072


,Exactitud,Precisión (macro),Recall (macro),F1 (macro)
Modelo actual (test),0.8946,0.8987,0.8998,0.8993


## 4. La idea: puntuar cada trozo de la reseña

El modelo actual ve cada reseña como tres bolsas de palabras. Eso funciona bien cuando la opinión está en la última oración, pero falla cuando la reseña tiene varias opiniones o termina con un comentario sin opinión (por ejemplo, "todavía no me decido"). Un modelo que entienda qué trozos de la reseña tienen opinión y cuál es la última debería equivocarse menos.

Para eso cada reseña se parte en **trozos**: primero en oraciones y luego cada oración se corta en los conectores de contraste que ya se usaban en el notebook principal ("pero", "aunque", "aun así", etc.). Cada trozo queda marcado con un 1 si viene después de un conector.

In [7]:
def separar_trozos(texto):
    """Parte cada oración en trozos usando los conectores de contraste.
    Devuelve una lista de (trozo, tras_conector); tras_conector vale 1 si el trozo viene después de un conector."""
    trozos = []
    for oracion in separar_oraciones(texto):
        partes = re.split(CONECTORES, quitar_tildes(oracion.lower()))
        for i, parte in enumerate(partes):
            if parte.strip():
                trozos.append((parte.strip(), int(i > 0)))
    return trozos


ejemplo = next(t for t in X_train['text'] if len(re.findall(CONECTORES, quitar_tildes(t.lower()))) > 0)
print('RESEÑA:\n', ejemplo, '\n')
for trozo, tras_conector in separar_trozos(ejemplo):
    print(f'  [tras conector: {tras_conector}]  {trozo}')

RESEÑA:
 Hace unos dias, lo uso a diario. La verdad, el material me reslto optimo la verdad. Con todo, el material se ve defectuoso la verdad. 

  [tras conector: 0]  hace unos dias, lo uso a diario.
  [tras conector: 0]  la verdad, el material me reslto optimo la verdad.
  [tras conector: 1]  , el material se ve defectuoso la verdad.


### 4.1 Etapa 1: un clasificador de trozos

La etapa 1 es un clasificador de trozos: TF-IDF con unigramas y bigramas y regresión logística, que devuelve la probabilidad de que un trozo sea `negativo`, `neutral` o `positivo`.

Como solo se conoce la etiqueta de la reseña completa y no la de cada trozo, **cada trozo hereda la etiqueta de su reseña**. Esto es ruidoso: una reseña positiva contiene trozos de contexto ("lo pedí un martes") que no son positivos. Aun así, las palabras que expresan opinión aparecen casi solo en trozos de una clase, mientras que el contexto aparece en todas, así que el modelo aprende a repartir sus probabilidades en el contexto y a concentrarlas en las palabras de opinión.

In [8]:
def entrenar_etapa1(textos, etiquetas, C):
    trozos, etiquetas_trozos = [], []
    for texto, etiqueta in zip(textos, etiquetas):
        for trozo, _ in separar_trozos(texto):
            trozos.append(limpiar_texto(trozo))
            etiquetas_trozos.append(etiqueta)    # cada trozo hereda la etiqueta de su reseña
    vectorizador = TfidfVectorizer(ngram_range=(1, 2), min_df=2)
    clasificador = LogisticRegression(C=C, max_iter=2000)
    clasificador.fit(vectorizador.fit_transform(trozos), etiquetas_trozos)
    return vectorizador, clasificador


vect_demo, clasif_demo = entrenar_etapa1(X_train['text'], y_train, C=3)
print('Clases del clasificador de trozos:', list(clasif_demo.classes_))


def puntuar_trozos(texto, vectorizador, clasificador):
    trozos = separar_trozos(texto)
    probabilidades = clasificador.predict_proba(vectorizador.transform([limpiar_texto(t) for t, _ in trozos]))
    tabla = pd.DataFrame(probabilidades, columns=['P(neg)', 'P(neu)', 'P(pos)']).round(2)
    tabla.insert(0, 'trozo', [t for t, _ in trozos])
    tabla.insert(1, 'tras_conector', [c for _, c in trozos])
    return tabla


puntuar_trozos(ejemplo, vect_demo, clasif_demo)

Clases del clasificador de trozos: [np.str_('negativo'), np.str_('neutral'), np.str_('positivo')]


,trozo,tras_conector,P(neg),P(neu),P(pos)
0,"hace unos dias, lo uso a diario.",0,0.46,0.09,0.44
1,"la verdad, el material me reslto optimo la verdad.",0,0.46,0.01,0.53
2,", el material se ve defectuoso la verdad.",1,0.89,0.00,0.11


En el ejemplo, el trozo "el material se ve defectuoso" recibe una probabilidad de 0,89 de ser negativo, y la reseña es negativa. Los trozos de contexto y el trozo "el material me resultó óptimo" reciben probabilidades repartidas (alrededor de 0,45 para cada extremo). Es decir, el clasificador de trozos es ruidoso y no se usa solo: sus puntajes son información adicional para la etapa 2.

### 4.2 Etapa 2: decidir la etiqueta de la reseña

La etapa 2 es una regresión logística (con los atributos estandarizados) que decide la etiqueta de la reseña a partir de 54 atributos:

1. Las **probabilidades de la regresión logística por bloques** del notebook principal (3 atributos).
2. Para cada uno de los **últimos 4 trozos**: sus 3 probabilidades y la marca de conector (16 atributos).
3. Un **resumen de los trozos**: la polaridad (probabilidad de positivo menos la de negativo) del último, el penúltimo y el primer trozo con opinión (se considera que un trozo tiene opinión si su probabilidad de ser neutral es menor a 0,5), cuántos trozos tienen opinión, el máximo de probabilidad de positivo y de negativo, el promedio de probabilidad de neutral y el número de trozos (8 atributos).
4. Las **probabilidades de Naive Bayes** sobre los mismos bloques (3 atributos).
5. Indicadores de **cómo empiezan las dos últimas oraciones con opinión**, para los 12 comienzos de oración más frecuentes, por ejemplo "la verdad" (24 atributos).

Con estos atributos la regresión logística puede aprender reglas que la bolsa de palabras no puede, por ejemplo: "si el último trozo no tiene opinión, mirar el anterior", o "una oración que empieza con determinado conector pesa más que otra".

Un cuidado importante es que **los atributos con los que se entrena la etapa 2 se calculan fuera de muestra**: cada reseña se puntúa con modelos que no la vieron al entrenarse (validación cruzada interna de 5 particiones). Si la etapa 1 se entrenara y se usara sobre las mismas reseñas, sus puntajes serían demasiado buenos y la etapa 2 aprendería a confiar demasiado en ellos.

## 5. El modelo en dos etapas

Todo el sistema se implementa como una clase, `ModeloDosEtapas`, que sigue la interfaz de scikit-learn (`fit`, `predict`, `predict_proba` y los hiperparámetros en el constructor). Por eso se puede usar con `GridSearchCV`, `cross_val_score` y `joblib` igual que cualquier otro modelo. Lo que hace `fit` es:

1. Calcular los atributos de entrenamiento fuera de muestra con 5 particiones (entrenando en cada una los modelos por bloques y la etapa 1).
2. Entrenar la etapa 2 con esos atributos.
3. Volver a entrenar los modelos por bloques y la etapa 1 con todos los datos, que son los que se usan al predecir reseñas nuevas.

In [9]:
K_ULTIMOS = 4          # cuántos de los últimos trozos se usan como atributos
N_INICIOS = 12         # cuántos comienzos de oración frecuentes se usan como atributos


def comienzo_oracion(oracion):
    palabras = re.sub(r'[^\w\s]', '', quitar_tildes(oracion.lower())).split()
    return ' '.join(palabras[:2])


def comienzos_frecuentes(textos, n=N_INICIOS):
    # Comienzos de oración más repetidos (sin contar la primera oración), por ejemplo "eso si" o "por otro"
    conteo = Counter(comienzo_oracion(o) for t in textos for o in separar_oraciones(t)[1:])
    return [c for c, _ in conteo.most_common(n)]


class ModeloDosEtapas(ClassifierMixin, BaseEstimator):
    """Clasificador de reseñas en dos etapas.

    Etapa 1: puntúa cada trozo de la reseña (probabilidad de ser negativo, neutral o positivo).
    Etapa 2: regresión logística que decide la etiqueta usando los puntajes de los últimos trozos,
    las probabilidades de los modelos por bloques y el comienzo de las oraciones con opinión.
    """

    def __init__(self, C_etapa1=3, C_etapa2=1, n_particiones=5, random_state=42):
        self.C_etapa1 = C_etapa1
        self.C_etapa2 = C_etapa2
        self.n_particiones = n_particiones
        self.random_state = random_state

    def _entrenar_componentes(self, textos, etiquetas):
        # Modelos por bloques (los mismos del notebook principal): regresión logística y Naive Bayes
        datos = pd.DataFrame({'text': textos.values})
        representacion = construir_representacion().fit(datos)
        matriz = representacion.transform(datos)
        regresion = LogisticRegression(C=10, penalty='l1', solver='liblinear', max_iter=3000,
                                       random_state=RANDOM_STATE).fit(matriz, etiquetas)
        bayes = MultinomialNB(alpha=0.3).fit(matriz, etiquetas)
        # Etapa 1
        vectorizador, clasificador = entrenar_etapa1(textos, etiquetas, self.C_etapa1)
        return {'representacion': representacion, 'regresion': regresion, 'bayes': bayes,
                'vectorizador': vectorizador, 'clasificador': clasificador}

    def _caracteristicas(self, textos, comp):
        datos = pd.DataFrame({'text': textos.values})
        matriz = comp['representacion'].transform(datos)
        prob_regresion = comp['regresion'].predict_proba(matriz)
        prob_bayes = comp['bayes'].predict_proba(matriz)

        # Se puntúan de una sola vez todos los trozos y todas las oraciones de todos los textos
        trozos_por_texto = [separar_trozos(t) for t in textos]
        oraciones_por_texto = [separar_oraciones(t) for t in textos]
        probs_trozos = comp['clasificador'].predict_proba(comp['vectorizador'].transform(
            [limpiar_texto(trozo) for lista in trozos_por_texto for trozo, _ in lista]))
        probs_oraciones = comp['clasificador'].predict_proba(comp['vectorizador'].transform(
            [limpiar_texto(o) for lista in oraciones_por_texto for o in lista]))

        filas = []
        ini_t = ini_o = 0
        for k in range(len(textos)):
            trozos, oraciones = trozos_por_texto[k], oraciones_por_texto[k]
            P = probs_trozos[ini_t:ini_t + len(trozos)]
            Po = probs_oraciones[ini_o:ini_o + len(oraciones)]
            ini_t += len(trozos)
            ini_o += len(oraciones)

            fila = list(prob_regresion[k])
            for j in range(1, K_ULTIMOS + 1):          # los últimos trozos: 3 probabilidades + marca de conector
                if j <= len(trozos):
                    fila += list(P[-j]) + [trozos[-j][1]]
                else:
                    fila += [0, 0, 0, 0]

            polaridad = P[:, 2] - P[:, 0]               # P(positivo) - P(negativo)
            con_opinion = np.where(P[:, 1] < 0.5)[0]    # trozos que no parecen neutrales
            fila += [polaridad[con_opinion[-1]] if len(con_opinion) > 0 else 0,
                     polaridad[con_opinion[-2]] if len(con_opinion) > 1 else 0,
                     polaridad[con_opinion[0]] if len(con_opinion) > 0 else 0,
                     len(con_opinion), P[:, 2].max(), P[:, 0].max(), P[:, 1].mean(), len(trozos)]

            fila += list(prob_bayes[k])
            # Comienzo de las dos últimas oraciones con opinión
            oraciones_con_opinion = np.where(Po[:, 1] < 0.5)[0]
            for pos in [-1, -2]:
                comienzo = ''
                if len(oraciones_con_opinion) >= -pos:
                    comienzo = comienzo_oracion(oraciones[oraciones_con_opinion[pos]])
                fila += [int(comienzo == c) for c in self.comienzos_]
            filas.append(fila)
        return np.array(filas)

    def fit(self, X, y):
        textos = pd.Series(np.asarray(X['text']))
        etiquetas = pd.Series(np.asarray(y))
        self.classes_ = np.array(sorted(etiquetas.unique()))
        self.comienzos_ = comienzos_frecuentes(textos)

        # Los atributos con los que se entrena la etapa 2 se calculan "fuera de muestra":
        # cada reseña se puntúa con modelos que no la vieron al entrenarse.
        particiones = StratifiedKFold(n_splits=self.n_particiones, shuffle=True, random_state=self.random_state)
        caracteristicas = None
        for idx_ent, idx_val in particiones.split(textos, etiquetas):
            comp = self._entrenar_componentes(textos.iloc[idx_ent], etiquetas.iloc[idx_ent])
            fila_val = self._caracteristicas(textos.iloc[idx_val], comp)
            if caracteristicas is None:
                caracteristicas = np.zeros((len(textos), fila_val.shape[1]))
            caracteristicas[idx_val] = fila_val

        self.etapa2_ = Pipeline([('escala', StandardScaler()),
                                 ('modelo', LogisticRegression(C=self.C_etapa2, max_iter=3000))])
        self.etapa2_.fit(caracteristicas, etiquetas)

        # Para predecir reseñas nuevas, los componentes se entrenan con todos los datos
        self.componentes_ = self._entrenar_componentes(textos, etiquetas)
        self.nombres_caracteristicas_ = self._nombres()
        return self

    def _nombres(self):
        nombres = ['regresion_neg', 'regresion_neu', 'regresion_pos']
        for j in range(1, K_ULTIMOS + 1):
            nombres += [f'trozo-{j}_neg', f'trozo-{j}_neu', f'trozo-{j}_pos', f'trozo-{j}_tras_conector']
        nombres += ['polaridad_ultima_opinion', 'polaridad_penultima_opinion', 'polaridad_primera_opinion',
                    'n_trozos_con_opinion', 'max_prob_pos', 'max_prob_neg', 'media_prob_neu', 'n_trozos']
        nombres += ['bayes_neg', 'bayes_neu', 'bayes_pos']
        nombres += [f'ultima_opinion_inicia_con:{c}' for c in self.comienzos_]
        nombres += [f'penultima_opinion_inicia_con:{c}' for c in self.comienzos_]
        return nombres

    def predict_proba(self, X):
        textos = pd.Series(np.asarray(X['text']))
        return self.etapa2_.predict_proba(self._caracteristicas(textos, self.componentes_))

    def predict(self, X):
        return self.classes_[self.predict_proba(X).argmax(axis=1)]

Dos valores quedaron fijos y no se buscaron con `GridSearchCV`: el número de trozos finales que se usan como atributos (4) y el número de comienzos de oración frecuentes (12). Se escogieron tras algunas pruebas exploratorias con validación cruzada.

## 6. Búsqueda de hiperparámetros

Se buscan los dos parámetros de regularización (`C`) del modelo: el de la etapa 1 y el de la etapa 2, con la misma validación cruzada y la misma métrica (F1-macro) que en el notebook principal. Una configuración tarda más que en el modelo actual porque cada ajuste entrena el sistema completo, incluida su validación cruzada interna.

In [10]:
grilla = {
    'C_etapa1': [1, 3, 10],       # regularización del clasificador de trozos
    'C_etapa2': [0.1, 1],         # regularización del clasificador de reseñas
}

busqueda = GridSearchCV(ModeloDosEtapas(), grilla, cv=kfold, scoring='f1_macro', n_jobs=-1)
busqueda.fit(X_train, y_train)

print('Mejores hiperparámetros:', busqueda.best_params_)
print(f'F1-macro en validación cruzada: {busqueda.best_score_:.4f}   (modelo actual: {cv_actual.mean():.4f})')

resultados = pd.DataFrame(busqueda.cv_results_)
resultados[['param_C_etapa1', 'param_C_etapa2', 'mean_test_score', 'std_test_score', 'rank_test_score']].sort_values('rank_test_score').round(4)

Mejores hiperparámetros: {'C_etapa1': 3, 'C_etapa2': 0.1}
F1-macro en validación cruzada: 0.9040   (modelo actual: 0.8992)


,param_C_etapa1,param_C_etapa2,mean_test_score,std_test_score,rank_test_score
2,3,0.1,0.9040,0.0074,1
4,10,0.1,0.9038,0.0069,2
5,10,1.0,0.9035,0.0065,3
3,3,1.0,0.9032,0.0065,4
0,1,0.1,0.9032,0.0083,5
1,1,1.0,0.9031,0.0079,6


La mejor configuración usa `C_etapa1 = 3` y `C_etapa2 = 0,1`, con un F1-macro de 0,9040 en validación cruzada, frente a 0,8992 del modelo actual. Las seis configuraciones están entre 0,9031 y 0,9040, es decir, la elección de `C` casi no importa; lo que sí se nota es que las seis superan al modelo actual.

## 7. Evaluación

### 7.1 Conjunto de prueba

In [11]:
mejor_modelo = busqueda.best_estimator_
pred_dos_etapas = pd.Series(mejor_modelo.predict(X_test), index=y_test.index)

tabla = pd.DataFrame({'Modelo actual': metricas(y_test, pred_actual),
                      'Dos etapas por oración': metricas(y_test, pred_dos_etapas)}).T
display(tabla.round(4))
print(classification_report(y_test, pred_dos_etapas, digits=3))

fig, ax = plt.subplots(1, 2, figsize=(10, 4), constrained_layout=True)
ConfusionMatrixDisplay.from_predictions(y_test, pred_actual, labels=ORDEN_CLASES, cmap='Blues', ax=ax[0], colorbar=False)
ax[0].set_title('Modelo actual')
ConfusionMatrixDisplay.from_predictions(y_test, pred_dos_etapas, labels=ORDEN_CLASES, cmap='Oranges', ax=ax[1], colorbar=False)
ax[1].set_title('Dos etapas por oración')
plt.show()

,Exactitud,Precisión (macro),Recall (macro),F1 (macro)
Modelo actual,0.8946,0.8987,0.8998,0.8993
Dos etapas por oración,0.8971,0.9021,0.9021,0.9021


              precision    recall  f1-score   support

    negativo      0.848     0.866     0.857       843
     neutral      0.996     0.997     0.997       715
    positivo      0.863     0.843     0.853       842

    accuracy                          0.897      2400
   macro avg      0.902     0.902     0.902      2400
weighted avg      0.897     0.897     0.897      2400



En el conjunto de prueba el modelo en dos etapas obtiene una exactitud de 0,8971 y un F1-macro de 0,9021, frente a 0,8946 y 0,8993 del modelo actual (+0,25 y +0,28 puntos). La clase `neutral` se mantiene casi perfecta y la mejora aparece en `negativo` (F1 de 0,857 contra 0,853).

La mejora es pequeña. Con 2.400 reseñas, el error estándar de una exactitud cercana a 0,90 es de unos 0,6 puntos, así que una diferencia de 0,25 puntos en un solo conjunto de prueba no es concluyente por sí sola.

### 7.2 Comprobación con particiones nuevas

Para no depender de un solo conjunto de prueba, se repite la comparación con particiones de validación cruzada nuevas (semilla 2024, distinta a la usada para elegir hiperparámetros) sobre las 9.600 reseñas de entrenamiento. Los dos modelos se evalúan en las mismas particiones, de modo que se pueden comparar partición por partición.

In [12]:
# Particiones de validación cruzada distintas a las usadas para elegir hiperparámetros (semilla nueva)
kfold_nuevo = StratifiedKFold(n_splits=5, shuffle=True, random_state=2024)

cv_base = cross_val_score(modelo_actual, X_train, y_train, cv=kfold_nuevo, scoring='accuracy', n_jobs=-1)
cv_nuevo = cross_val_score(clone(mejor_modelo), X_train, y_train, cv=kfold_nuevo, scoring='accuracy', n_jobs=-1)

comparacion = pd.DataFrame({'Modelo actual': cv_base, 'Dos etapas': cv_nuevo})
comparacion['Diferencia'] = comparacion['Dos etapas'] - comparacion['Modelo actual']
comparacion.index = [f'Partición {i + 1}' for i in range(5)]
display(comparacion.round(4))
print(f'Exactitud media -> modelo actual: {cv_base.mean():.4f} | dos etapas: {cv_nuevo.mean():.4f}')
print(f'Diferencia media: {comparacion["Diferencia"].mean():+.4f}  (mejora en {(comparacion["Diferencia"] > 0).sum()} de 5 particiones)')

,Modelo actual,Dos etapas,Diferencia
Partición 1,0.9010,0.9042,0.0031
Partición 2,0.8958,0.9062,0.0104
Partición 3,0.8880,0.8896,0.0016
Partición 4,0.8870,0.8917,0.0047
Partición 5,0.8948,0.8964,0.0016


Exactitud media -> modelo actual: 0.8933 | dos etapas: 0.8976
Diferencia media: +0.0043  (mejora en 5 de 5 particiones)


El modelo en dos etapas mejora en las 5 particiones, con una diferencia media de +0,43 puntos de exactitud (entre +0,16 y +1,04). El resultado concuerda con el del conjunto de prueba: una mejora pequeña pero consistente en el signo, de entre 0,25 y 0,45 puntos.

## 8. ¿Qué cambió respecto al modelo actual?

### 8.1 Reseñas que cambian de predicción

In [13]:
corrige = (pred_actual != y_test) & (pred_dos_etapas == y_test)
empeora = (pred_actual == y_test) & (pred_dos_etapas != y_test)
print(f'Reseñas que el modelo actual fallaba y ahora acierta: {corrige.sum()}')
print(f'Reseñas que el modelo actual acertaba y ahora falla:  {empeora.sum()}')
print(f'Balance neto: {corrige.sum() - empeora.sum():+d} de {len(y_test)} reseñas de prueba')

Reseñas que el modelo actual fallaba y ahora acierta: 40
Reseñas que el modelo actual acertaba y ahora falla:  34
Balance neto: +6 de 2400 reseñas de prueba


In [14]:
modelo_etapa1 = (mejor_modelo.componentes_['vectorizador'], mejor_modelo.componentes_['clasificador'])

for i in y_test[corrige].index[:3]:
    print(f'REAL: {y_test[i]} | modelo actual: {pred_actual[i]} | dos etapas: {pred_dos_etapas[i]}')
    display(puntuar_trozos(X_test.loc[i, 'text'], *modelo_etapa1))

REAL: negativo | modelo actual: positivo | dos etapas: negativo


,trozo,tras_conector,P(neg),P(neu),P(pos)
0,la ergonomia vale totalmente la pena.,0,0.32,0.01,0.67
1,"de paso, le pondria una estrella al boton de encendido.",0,0.59,0.02,0.40
2,todavia no me decido.,0,0.45,0.04,0.51


REAL: positivo | modelo actual: negativo | dos etapas: positivo


,trozo,tras_conector,P(neg),P(neu),P(pos)
0,"lo compre a mediados de ano porque lo necesitaba para el cuarto donde trabajo desde casa, y el sistema de sonido me ...",0,0.49,0.00,0.51
1,"el envio tardo como cuatro dias, nada fuera de lo normal.",0,0.69,0.10,0.21
2,"por cierto, cada peso que pague por el ventilador valio la pena, lo tengo corriendo casi todas las tardes.",0,0.27,0.00,0.73
3,cada quien que juzgue.,0,0.44,0.01,0.55


REAL: negativo | modelo actual: positivo | dos etapas: negativo


,trozo,tras_conector,P(neg),P(neu),P(pos)
0,compre este procesador hace un mes por internet y me llego en cinco dias.,0,0.47,0.08,0.45
1,cada peso que pague por el procesador valio la pena.,0,0.45,0.02,0.53
2,lo instale en la torre que armo en mi cuarto.,0,0.24,0.24,0.52
3,la verdad es que no pare de tener problemas con la duracion de la carga.,0,0.48,0.00,0.52


En las 2.400 reseñas de prueba, el modelo en dos etapas acierta 40 reseñas que el modelo actual fallaba y falla 34 que antes acertaba: un balance neto de +6 reseñas (0,25%). Es decir, cambia bastante de opinión sobre muchas reseñas, pero casi todos los cambios se compensan.

Los dos primeros ejemplos corregidos terminan en una frase sin opinión ("todavía no me decido", "cada quien que juzgue") precedida por una opinión. En el modelo actual, el bloque de última oración solo contenía esa frase vacía de opinión, y el modelo en dos etapas parece poder mirar el trozo anterior con opinión. El tercer ejemplo muestra el límite de la etapa 1: el trozo "no paré de tener problemas con la duración de la carga" recibe una probabilidad de 0,52 de ser positivo, es decir, el clasificador de trozos todavía se equivoca con negaciones, y el modelo acierta apoyándose en otras señales.

### 8.2 Reseñas con cierre ambiguo

In [15]:
def clave_oracion(oracion):
    return re.sub(r'[^\w\s]', '', quitar_tildes(oracion.lower())).strip()


finales = Counter(clave_oracion(separar_oraciones(t)[-1]) for t in X_train['text'])
cierres = {o for o, n in finales.items() if n >= 10}
termina_en_cierre = X_test['text'].map(lambda t: clave_oracion(separar_oraciones(t)[-1]) in cierres)
es_polar = y_test != 'neutral'

grupos = pd.DataFrame({
    'reseñas positivas/negativas': [(es_polar & termina_en_cierre).sum(), (es_polar & ~termina_en_cierre).sum()],
    'exactitud modelo actual': [(pred_actual == y_test)[es_polar & termina_en_cierre].mean(),
                                (pred_actual == y_test)[es_polar & ~termina_en_cierre].mean()],
    'exactitud dos etapas': [(pred_dos_etapas == y_test)[es_polar & termina_en_cierre].mean(),
                             (pred_dos_etapas == y_test)[es_polar & ~termina_en_cierre].mean()],
}, index=['Terminan en un cierre repetido', 'Terminan en otra oración'])
grupos.round(3)

,reseñas positivas/negativas,exactitud modelo actual,exactitud dos etapas
Terminan en un cierre repetido,198,0.460,0.485
Terminan en otra oración,1487,0.902,0.904


En las reseñas que terminan en un cierre ambiguo la exactitud pasa de 0,460 a 0,485, y en el resto de 0,902 a 0,904. En ambos casos son diferencias de unas pocas reseñas (198 y 1.487 reseñas respectivamente), por lo que no se puede afirmar que el modelo resuelva el problema de los cierres ambiguos. Era de esperarse: en ese grupo parece haber poca información en el texto para decidir.

### 8.3 Qué atributos usa la etapa 2

In [16]:
modelo_etapa2 = mejor_modelo.etapa2_.named_steps['modelo']
coeficientes = pd.DataFrame(modelo_etapa2.coef_.T, index=mejor_modelo.nombres_caracteristicas_,
                            columns=mejor_modelo.classes_)

for clase in ['positivo', 'negativo']:
    print(f'\nAtributos con más peso hacia "{clase}":')
    display(coeficientes[clase].sort_values(ascending=False).head(8).round(2).to_frame('coeficiente'))


Atributos con más peso hacia "positivo":


,coeficiente
regresion_pos,0.72
max_prob_pos,0.47
bayes_pos,0.32
trozo-2_neg,0.29
trozo-1_neg,0.29
n_trozos,0.16
trozo-1_tras_conector,0.13
ultima_opinion_inicia_con:la verdad,0.11



Atributos con más peso hacia "negativo":


,coeficiente
regresion_neg,0.56
bayes_neg,0.51
max_prob_neg,0.42
trozo-2_pos,0.31
trozo-1_pos,0.26
trozo-3_pos,0.21
trozo-4_pos,0.20
n_trozos_con_opinion,0.16


La etapa 2 se apoya sobre todo en las probabilidades de los modelos por bloques (`regresion_*` y `bayes_*`) y en el máximo de probabilidad de los trozos (`max_prob_*`). Los atributos de los últimos trozos también tienen peso, y aparece uno de los comienzos de oración ("la verdad"). Como las tres probabilidades de un trozo suman 1, los coeficientes de atributos correlacionados pueden compensarse entre sí, así que conviene leer el modelo en conjunto y no atributo por atributo.

El resultado importante es que la etapa 2 mantiene casi toda la información del modelo actual y le suma una pequeña corrección basada en los trozos.

## 9. Modelo final y predicciones para la competencia

La configuración elegida se vuelve a entrenar con las 12.000 reseñas etiquetadas y se usa para predecir las 3.000 reseñas de `eval.csv`. Después se compara con el envío del modelo actual para ver cuántas predicciones cambian.

In [17]:
modelo_final = clone(mejor_modelo)
modelo_final.fit(X, y)

pred_eval = modelo_final.predict(evaluacion[['text']])
envio = pd.DataFrame({'id': evaluacion['id'], 'answer': pred_eval})
envio.to_csv(os.path.join(RUTA_ENVIOS, f'submission_{NOMBRE_MODELO}.csv'), index=False)

print('Mismas columnas que sample_submission:', list(envio.columns) == list(muestra_envio.columns))
print('Mismos ids y en el mismo orden:', (envio['id'] == muestra_envio['id']).all())
display(pd.DataFrame({'train (real)': train['label'].value_counts(normalize=True),
                      'eval (predicho)': envio['answer'].value_counts(normalize=True)}).reindex(ORDEN_CLASES).round(3))

Mismas columnas que sample_submission: True
Mismos ids y en el mismo orden: True


,train (real),eval (predicho)
negativo,0.351,0.360
neutral,0.298,0.277
positivo,0.351,0.363


In [18]:
envio_actual = pd.read_csv(os.path.join(RUTA_ENVIOS, 'submission_regresion_logistica_bloques.csv'))
distintas = envio['answer'] != envio_actual['answer']
print(f'Predicciones distintas respecto al envío del modelo actual: {distintas.sum()} de {len(envio)} ({distintas.mean():.1%})')
pd.crosstab(envio_actual['answer'][distintas], envio['answer'][distintas], rownames=['modelo actual'], colnames=['dos etapas'])

Predicciones distintas respecto al envío del modelo actual: 120 de 3000 (4.0%)


dos etapas,negativo,positivo
modelo actual,,
negativo,0,51
neutral,6,5
positivo,58,0


In [19]:
ruta_modelo = os.path.join(RUTA_MODELOS, f'{NOMBRE_MODELO}.joblib')
joblib.dump(modelo_final, ruta_modelo)

modelo_cargado = joblib.load(ruta_modelo)
print('El modelo guardado reproduce exactamente las mismas predicciones:',
      (modelo_cargado.predict(evaluacion[['text']]) == pred_eval).all())

El modelo guardado reproduce exactamente las mismas predicciones: True


El modelo en dos etapas cambia 120 de las 3.000 predicciones (4,0%): 51 pasan de `negativo` a `positivo`, 58 de `positivo` a `negativo` y 11 involucran a `neutral`. Con las mejoras medidas (entre 0,25 y 0,45 puntos), se espera que el balance neto sea de alrededor de 10 reseñas a favor, es decir, que el puntaje en Kaggle suba unas pocas décimas de punto. Es una estimación y no una garantía, porque el ruido de una muestra de 3.000 reseñas (cerca de 0,6 puntos) es mayor que esa mejora.

## 10. Conclusiones

- **La idea funciona, pero poco.** El modelo en dos etapas sube el F1-macro en validación cruzada de 0,8992 a 0,9040, la exactitud en el conjunto de prueba de 0,8946 a 0,8971 y mejora en las 5 particiones de una validación cruzada con semilla nueva (+0,43 puntos de exactitud). Es una mejora real y consistente, pero de menos de medio punto.
- **El cuello de botella son los datos y la etapa 1.** Los cierres ambiguos siguen sin resolverse (0,485 de exactitud) y el clasificador de trozos, entrenado con etiquetas heredadas de la reseña, es ruidoso (por ejemplo, con negaciones). La etapa 2 mantiene casi toda la información del modelo actual y le suma una corrección pequeña.
- **Cuesta más.** Es una clase propia, con 54 atributos, un entrenamiento que incluye validación cruzada interna y unos 7 minutos de ejecución. Es más difícil de explicar que el modelo del notebook principal y se aleja un poco de lo visto en clase (encadenar modelos y usar etiquetas heredadas).
- **Recomendación.** Para el ranking de Kaggle, una ganancia de unas décimas queda dentro del ruido de la muestra. Se puede enviar este modelo como un envío adicional, pero el modelo que se entregue en Bloque Neón debe corresponder al mejor envío de Kaggle del grupo.

## 11. Uso de herramientas de IA generativa

**Declaración del uso.** Se utilizó Claude Code (modelo Claude Sonnet 5.5, de Anthropic) como apoyo para: proponer y generar un primer esqueleto del código de este notebook, depurar errores y redactar las explicaciones.

**Prompts utilizados.**

1. *"implementemos las 3 oportunidades de mejora en 3 jupyters nuevos"*, después de que la herramienta resumiera las oportunidades de mejora del modelo del notebook principal (modelo por oración, validación cruzada repetida y ajuste fino de la regresión logística).

**Análisis crítico del resultado.**

[Completar por el equipo: ¿qué partes del código y de las explicaciones generadas fueron correctas y útiles? ¿qué errores, imprecisiones o limitaciones se identificaron? ¿qué decisiones técnicas se cambiaron respecto a lo que propuso la herramienta y por qué? ¿qué conceptos del curso permitieron evaluar o corregir la respuesta?]

**Aportes propios del equipo.**

[Completar por el equipo: qué se desarrolló, modificó o decidió, qué ajustes se hicieron al código y a las explicaciones, y qué se aprendió del proceso.]

## 12. Cómo replicar los resultados

1. Crear un entorno con Python 3.9 e instalar las librerías con `pip install -r requirements.txt` (están en la raíz del proyecto).
2. Mantener la estructura de carpetas del proyecto: este notebook está en `notebooks/`, los datos en `data/`. El notebook se ejecuta desde la carpeta `notebooks/`.
3. Ejecutar todas las celdas en orden (`Restart & Run All`). Tarda aproximadamente 7 minutos.
4. Al terminar se generan `models/dos_etapas_por_oracion.joblib` y `submissions/submission_dos_etapas_por_oracion.csv`.

Todas las fuentes de aleatoriedad usan semillas fijas, por lo que las cifras se reproducen exactamente con las versiones de `requirements.txt`. Para cargar el modelo guardado fuera del notebook hay que ejecutar antes las celdas que definen las funciones y la clase `ModeloDosEtapas` usadas por el pipeline.